In [3]:
import json
import random
import sys
import requests

# ==============================================================================
# CONFIGURACIÓN DE PARÁMETROS DE USUARIO
# Ajuste estos parámetros para que coincidan con su entorno local/remoto de CouchDB.
# ==============================================================================
COUCHDB_HOST = "127.0.0.1"  # IP o Hostname de CouchDB
COUCHDB_PORT = 5984  # Puerto por defecto de CouchDB
COUCHDB_USER = "Rojaja15"  # Su usuario de CouchDB
COUCHDB_PASS = "Javier15"  # Su contraseña de CouchDB
DATABASE_NAME = "mercadotico360"  # Nombre de la base de datos objetivo

BATCH_SIZE = 5000  # Tamaño del lote para inserción masiva (_bulk_docs)

# Construcción de URLs base
BASE_URL = f"http://{COUCHDB_HOST}:{COUCHDB_PORT}"
BULK_URL = f"{BASE_URL}/{DATABASE_NAME}/_bulk_docs"
AUTH = (COUCHDB_USER, COUCHDB_PASS)


# ==============================================================================
# FUNCIONES AUXILIARES DE GESTIÓN Y CARGA
# ==============================================================================
def verificar_o_crear_bd():
  """Verifica la conectividad con CouchDB y asegura que la base de datos exista."""
  try:
    db_url = f"{BASE_URL}/{DATABASE_NAME}"
    res = requests.get(db_url, auth=AUTH)

    if res.status_code == 200:
      print(f"[OK] La base de datos '{DATABASE_NAME}' ya existe.")
    elif res.status_code == 404:
      create_res = requests.put(db_url, auth=AUTH)
      if create_res.status_code in [201, 202]:
        print(f"[OK] Base de datos '{DATABASE_NAME}' creada exitosamente.")
      else:
        print(
            f"[ERROR] No se pudo crear la BD: {create_res.status_code} -"
            f" {create_res.text}"
        )
        sys.exit(1)
    else:
      print(
          f"[ERROR] Error al verificar BD: {res.status_code} - {res.text}"
      )
      sys.exit(1)
  except requests.exceptions.ConnectionError:
    print(
        f"[ERROR CRÍTICO] No fue posible conectar con CouchDB en"
        f" {BASE_URL}. Asegúrese de que el servicio esté ejecutándose."
    )
    sys.exit(1)


def enviar_lote(documentos):
  """Envía un conjunto de documentos a CouchDB mediante el endpoint /_bulk_docs."""
  headers = {"Content-Type": "application/json"}
  payload = {"docs": documentos}

  try:
    res = requests.post(BULK_URL, json=payload, auth=AUTH, headers=headers)

    if res.status_code in [200, 201]:
      print(f" -> Lote de {len(documentos)} documentos insertado con éxito.")
    else:
      print(f" -> [ERROR] Falló la inserción: {res.status_code} - {res.text}")
  except Exception as e:
    print(f" -> [EXCEPCIÓN] Ocurrió un error al enviar el lote: {str(e)}")


# ==============================================================================
# GENERACIÓN DE DATOS SINTÉTICOS CON ATRIBUTOS HETEROGÉNEOS (CASO 2)
# ==============================================================================
def generar_productos(cantidad=25000):
  """Genera productos distribuidos en al menos 8 categorías con atributos variables (Requisito 2)."""
  categorias = [
      "Electrónica",
      "Hogar",
      "Moda",
      "Deportes",
      "Alimentación",
      "Juguetería",
      "Belleza",
      "Automotriz",
  ]
  productos = []

  for i in range(1, cantidad + 1):
    cat = random.choice(categorias)
    doc = {
        "tipo_doc": "producto",
        "producto_id": f"PROD-{i:06d}",
        "nombre": f"Producto {cat} {i}",
        "categoria": cat,
        "precio": round(random.uniform(5.0, 1500.0), 2),
        "stock": random.randint(0, 500),
        "atributos": {},
    }

    # Asignación de atributos específicos heterogéneos según la categoría
    if cat == "Electrónica":
      doc["atributos"] = {
          "voltaje": random.choice(["110V", "220V", "Batería recargable"]),
          "garantia_meses": random.choice([6, 12, 24]),
          "marca": f"TechBrand-{random.randint(1, 50)}",
      }
    elif cat == "Moda":
      doc["atributos"] = {
          "talla": random.choice(["XS", "S", "M", "L", "XL"]),
          "color": random.choice(["Rojo", "Azul", "Negro", "Blanco"]),
          "material": random.choice(["Algodón", "Poliéster", "Cuero"]),
      }
    elif cat == "Alimentación":
      doc["atributos"] = {
          "peso_kg": round(random.uniform(0.1, 5.0), 2),
          "perecedero": random.choice([True, False]),
          "registro_sanitario": f"C-CR-{random.randint(1000, 9999)}",
      }
    elif cat == "Deportes":
      doc["atributos"] = {
          "disciplina": random.choice(
              ["Fútbol", "Ciclismo", "Natación", "Gym"]
          ),
          "impermeable": random.choice([True, False]),
      }
    elif cat == "Hogar":
      doc["atributos"] = {
          "dimensiones_cm": "50x40x30",
          "requiere_ensamblaje": random.choice([True, False]),
      }
    elif cat == "Automotriz":
      doc["atributos"] = {
          "compatibilidad_modelo": f"Modelo-{random.randint(2015, 2026)}",
          "tipo_repuesto": random.choice(
              ["Motor", "Frenos", "Accesorios", "Eléctrico"]
          ),
      }
    elif cat == "Belleza":
      doc["atributos"] = {
          "tipo_piel": random.choice(["Grases", "Seca", "Mixta", "Todo tipo"]),
          "contenido_ml": random.choice([50, 100, 250, 500]),
      }
    elif cat == "Juguetería":
      doc["atributos"] = {
          "edad_minima_recomendada": random.choice([3, 6, 8, 12]),
          "incluye_baterias": random.choice([True, False]),
      }

    productos.append(doc)
  return productos


def generar_clientes(cantidad=10000):
  """Genera clientes sintéticos con ubicaciones en Costa Rica."""
  provincias = [
      "San José",
      "Alajuela",
      "Cartago",
      "Heredia",
      "Guanacaste",
      "Puntarenas",
      "Limón",
  ]
  clientes = []
  for i in range(1, cantidad + 1):
    doc = {
        "tipo_doc": "cliente",
        "cliente_id": f"CLI-{i:06d}",
        "nombre": f"Cliente {i}",
        "provincia": random.choice(provincias),
        "email": f"usuario{i}@ejemplo.cr",
        "fecha_registro": "2026-01-15",
    }
    clientes.append(doc)
  return clientes


def generar_pedidos(
    cantidad=50000, max_cliente=10000, max_producto=25000, lista_productos=None
):
  """Genera pedidos con estructuras anidadas (líneas de detalle) que conservan la información del producto."""
  estados = ["completado", "pendiente", "cancelado"]
  pedidos = []

  for i in range(1, cantidad + 1):
    cant_lineas = random.randint(1, 4)
    lineas_detalle = []
    monto_total = 0.0

    for _ in range(cant_lineas):
      prod_idx = random.randint(1, max_producto)
      prod_id = f"PROD-{prod_idx:06d}"

      # Recuperar precio snapshot para conservar información histórica
      precio_unitario = (
          lista_productos[prod_idx - 1]["precio"]
          if lista_productos
          else round(random.uniform(5.0, 500.0), 2)
      )
      unidades = random.randint(1, 3)
      subtotal = round(precio_unitario * unidades, 2)
      monto_total += subtotal

      lineas_detalle.append({
          "producto_id": prod_id,
          "nombre_snapshot": f"Producto {prod_idx}",
          "cantidad": unidades,
          "precio_unitario": precio_unitario,
          "subtotal": subtotal,
      })

    doc = {
        "tipo_doc": "pedido",
        "pedido_id": f"PED-{i:06d}",
        "cliente_id": f"CLI-{random.randint(1, max_cliente):06d}",
        "fecha_pedido": f"2026-0{random.randint(1, 9)}-{random.randint(10, 28)}",
        "lineas_detalle": lineas_detalle,
        "monto_total": round(monto_total, 2),
        "estado": random.choice(estados),
    }
    pedidos.append(doc)
  return pedidos


# ==============================================================================
# EJECUCIÓN DEL FLUJO
# ==============================================================================
if __name__ == "__main__":
  print("=== INICIANDO SIMULACIÓN Y CARGA DE DATOS (MERCADOTICO 360) ===")

  # 1. Crear / Validar Base de Datos en CouchDB
  verificar_o_crear_bd()

  # 2. Generación de datos sintéticos
  print("\nGenerando productos heterogéneos (8 categorías)...")
  docs_productos = generar_productos(25000)

  print("Generando clientes...")
  docs_clientes = generar_clientes(10000)

  print("Generando pedidos con líneas de detalle anidadas...")
  docs_pedidos = generar_pedidos(
      50000, 10000, 25000, lista_productos=docs_productos
  )

  todos_los_documentos = docs_productos + docs_clientes + docs_pedidos
  total_docs = len(todos_los_documentos)
  print(f"\nTotal general de documentos generados: {total_docs}")

  # 3. Carga masiva por lotes
  print(f"Iniciando inserción masiva en bloques de {BATCH_SIZE} documentos...")
  for i in range(0, total_docs, BATCH_SIZE):
    lote = todos_los_documentos[i : i + BATCH_SIZE]
    print(
        f"Enviando lote {i // BATCH_SIZE + 1} ({i + 1} a"
        f" {min(i + BATCH_SIZE, total_docs)})..."
    )
    enviar_lote(lote)

  print("\n=== CARGA COMPLETA Y SIMULACIÓN CONCLUIDA CON ÉXITO ===")

=== INICIANDO SIMULACIÓN Y CARGA DE DATOS (MERCADOTICO 360) ===
[OK] La base de datos 'mercadotico360' ya existe.

Generando productos heterogéneos (8 categorías)...
Generando clientes...
Generando pedidos con líneas de detalle anidadas...

Total general de documentos generados: 85000
Iniciando inserción masiva en bloques de 5000 documentos...
Enviando lote 1 (1 a 5000)...
 -> Lote de 5000 documentos insertado con éxito.
Enviando lote 2 (5001 a 10000)...
 -> Lote de 5000 documentos insertado con éxito.
Enviando lote 3 (10001 a 15000)...
 -> Lote de 5000 documentos insertado con éxito.
Enviando lote 4 (15001 a 20000)...
 -> Lote de 5000 documentos insertado con éxito.
Enviando lote 5 (20001 a 25000)...
 -> Lote de 5000 documentos insertado con éxito.
Enviando lote 6 (25001 a 30000)...
 -> Lote de 5000 documentos insertado con éxito.
Enviando lote 7 (30001 a 35000)...
 -> Lote de 5000 documentos insertado con éxito.
Enviando lote 8 (35001 a 40000)...
 -> Lote de 5000 documentos insertado